In [71]:
%conda install -c conda-forge lightgbm


In [ ]:
%pip install xgboost imbalanced-learn -q

In [12]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
#import xgboost as xgb
from tqdm import tqdm
import numpy as np

In [4]:
df_txn = pd.read_csv("../data/acct_transaction.csv")
df_alert = pd.read_csv("../data/acct_alert.csv")

## 特徵工程

In [6]:
def build_features(df_txn):
    acct_features = df_txn.groupby("from_acct").agg(
        txn_count=("txn_amt", "count"),
        txn_sum=("txn_amt", "sum"),
        txn_mean=("txn_amt", "mean"),
        txn_max=("txn_amt", "max"),
        to_acct_unique=("to_acct", "nunique"),
    ).reset_index()

    # 金額波動 & 大額比例
    acct_features["txn_amt_std"] = df_txn.groupby("from_acct")["txn_amt"].std().fillna(0)
    acct_features["txn_large_ratio"] = (
        df_txn.assign(is_large=df_txn["txn_amt"] > 100000)
        .groupby("from_acct")["is_large"].mean()
    )

    # 通路類型 (one-hot)
    channel_dummies = pd.get_dummies(df_txn["channel_type"], prefix="channel")
    df_txn = pd.concat([df_txn, channel_dummies], axis=1)
    acct_channel = df_txn.groupby("from_acct")[channel_dummies.columns].mean()
    acct_features = acct_features.merge(acct_channel, on="from_acct", how="left")

    # 跨行比例
    acct_features["cross_bank_ratio"] = (
        df_txn.assign(cross=(df_txn["to_acct_type"] == "02"))
        .groupby("from_acct")["cross"].mean()
    )

    # 活躍天數 + 夜間比例
    df_txn["txn_time"] = pd.to_datetime(
        df_txn["txn_time"], format="%H%M", errors="coerce"
    ).dt.hour
    df_txn["night"] = df_txn["txn_time"].apply(lambda h: (h >= 22) or (h < 6))
    acct_features["active_days"] = df_txn.groupby("from_acct")["txn_date"].nunique()
    acct_features["night_ratio"] = df_txn.groupby("from_acct")["night"].mean()

    # 熵
    counts = df_txn.groupby(["from_acct", "to_acct"]).size().reset_index(name="cnt")
    total = counts.groupby("from_acct")["cnt"].transform("sum")
    counts["p"] = counts["cnt"] / total
    entropy_df = counts.groupby("from_acct").apply(
        lambda g: -(g["p"] * np.log2(g["p"])).sum()
    )
    acct_features["to_acct_entropy"] = entropy_df

    return acct_features


In [7]:
def build_train_data(acct_features, df_alert):
    df_alert = df_alert.rename(columns={"acct": "from_acct"})
    train_data = acct_features.merge(df_alert, on="from_acct", how="left")
    train_data["label"] = train_data["event_date"].notna().astype(int)
    train_data["event_date"] = train_data["event_date"].fillna(-1).astype(int)
    train_data = train_data.fillna(0)  # 缺失補 0
    return train_data

In [16]:
from sklearn.model_selection import train_test_split


def prepare_dmatrix(train_data):
    X = train_data.drop(columns=["from_acct", "event_date", "label"], errors="ignore")
    y = train_data["label"]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    
    # 獲取特徵名稱列表
    feature_names = X_train.columns.tolist()

    return X_train, X_val, y_train, y_val

In [13]:
from tqdm import tqdm
from xgboost.callback import TrainingCallback


class TQDMCallback(TrainingCallback):
    def __init__(self, total):
        self.pbar = tqdm(total=total, desc="Training XGBoost")

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False

    def after_training(self, model):
        self.pbar.close()
        return model


def train_xgb(dtrain, dval, scale_pos_weight):
    params = {
        "objective": "binary:logistic",
        "max_depth": 6,
        "eta": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": ["auc", "aucpr"],
        "seed": 42,
    }
    num_round = 300
    evals = [(dtrain, "train"), (dval, "val")]

    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=num_round,
        evals=evals,
        verbose_eval=False,
        callbacks=[TQDMCallback(num_round)],
    )
    return bst

ModuleNotFoundError: No module named 'xgboost'

In [14]:
# 特徵工程
acct_features = build_features(df_txn)

# 合併標籤
train_data = build_train_data(acct_features, df_alert)



/tmp/ipykernel_7330/3220669882.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  entropy_df = counts.groupby("from_acct").apply(


In [17]:
# 切分 + DMatrix
X_train, X_val, y_train, y_val = prepare_dmatrix(train_data)

# 計算正負樣本比例
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print("scale_pos_weight:", scale_pos_weight)


scale_pos_weight: 1066.6205211726385


In [57]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
)

# 驗證集預測
y_proba = bst.predict(dval)
y_pred = (y_proba > 0.5).astype(int)

print("分類報告 (threshold=0.5):")
print(classification_report(y_val, y_pred, digits=4))

print("ROC-AUC:", roc_auc_score(y_val, y_proba))
print("PR-AUC:", average_precision_score(y_val, y_proba))  # 對 imbalanced data 特別重要

In [58]:
for th in [0.5, 0.3, 0.1]:
    y_pred_th = (y_proba > th).astype(int)
    print(f"\nThreshold = {th}")
    print(classification_report(y_val, y_pred_th, digits=4))

In [59]:
import matplotlib.pyplot as plt


# ========= 特徵重要性 =========
importances = bst.get_score(importance_type="gain")
sorted_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)

print("Top features by importance:")
for feat, score in sorted_features[:20]:
    print(f"{feat}: {score}")

# 畫圖會直接用原始欄位名
xgb.plot_importance(bst, importance_type="gain", max_num_features=20)
plt.show()

In [62]:
print(train_data.groupby("label")["to_acct_unique"].describe())
import seaborn as sns

sns.boxplot(x="label", y="to_acct_unique", data=train_data)

In [63]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.linspace(0.01, 0.5, 50)
f1_scores = [f1_score(y_val, (y_proba > th).astype(int)) for th in thresholds]
best_th = thresholds[np.argmax(f1_scores)]
print("最佳 threshold:", best_th, "F1:", max(f1_scores))

In [67]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, f1_score, roc_auc_score
import xgboost as xgb
from tqdm import tqdm

# ========= 特徵 / 標籤 =========
X = train_data.drop(columns=["from_acct", "event_date", "label"], errors="ignore")
y = train_data["label"]

# ========= 切分資料集 =========
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ========= SMOTE 過採樣 =========
print("Before SMOTE:", y_train.value_counts())
smote = SMOTE(sampling_strategy=0.5, random_state=42)
# 讓正樣本數量達到負樣本的 50%
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("After SMOTE:", y_train_res.value_counts())

# ========= DMatrix =========
dtrain = xgb.DMatrix(
    X_train_res, label=y_train_res, feature_names=X_train.columns.tolist()
)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=X_val.columns.tolist())

# ========= XGBoost 參數 =========
params = {
    "objective": "binary:logistic",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "aucpr",
    "seed": 42,
}

num_round = 300
evals = [(dtrain, "train"), (dval, "val")]

# ========= 訓練 =========
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    evals=evals,
    verbose_eval=50,
)

# ========= 驗證 =========
y_proba = bst.predict(dval)

best_f1, best_th = 0, 0
for th in [i / 100 for i in range(1, 100)]:
    y_pred = (y_proba > th).astype(int)
    f1 = f1_score(y_val, y_pred)
    if f1 > best_f1:
        best_f1, best_th = f1, th

print(f"最佳 threshold: {best_th:.2f}, F1: {best_f1:.4f}")
print("ROC-AUC:", roc_auc_score(y_val, y_proba))

# ========= 在最佳 threshold 下的分類報告 =========
y_pred = (y_proba > best_th).astype(int)
print(classification_report(y_val, y_pred))

In [68]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# ========= 模型輸出 =========
y_proba = bst.predict(dval)  # validation set 機率
y_true = y_val.values


# ========= Top-k 評估函數 =========
def evaluate_topk(y_true, y_proba, k_list=[100, 500, 1000, 2000]):
    results = {}
    sorted_idx = np.argsort(-y_proba)  # 機率由高到低排序
    for k in k_list:
        topk_idx = sorted_idx[:k]
        y_pred = np.zeros_like(y_true)
        y_pred[topk_idx] = 1
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        results[k] = {"precision": precision, "recall": recall, "f1": f1}
    return results


# ========= 測試不同 k =========
k_results = evaluate_topk(y_true, y_proba, k_list=[100, 500, 1000, 2000, 5000])
for k, metrics in k_results.items():
    print(f"\nTop-{k}")
    print(metrics)

In [73]:
from tqdm import tqdm

# ========== 建立帳戶特徵 ==========
print("Building features...")
acct_features = (
    df_txn.groupby("from_acct")
    .agg(
        txn_count=("txn_amt", "count"),
        txn_sum=("txn_amt", "sum"),
        txn_mean=("txn_amt", "mean"),
        txn_max=("txn_amt", "max"),
        to_acct_unique=("to_acct", "nunique"),
        channel_unique=("channel_type", "nunique"),
    )
    .reset_index()
)

# ========== 加入大額比例 ==========
print("Adding large transaction ratio...")
acct_features["txn_amt_std"] = df_txn.groupby("from_acct")["txn_amt"].std().fillna(0)
acct_features["txn_large_ratio"] = (
    df_txn.assign(is_large=df_txn["txn_amt"] > 100000)
    .groupby("from_acct")["is_large"]
    .mean()
)

In [74]:
# ========== 通路 one-hot ==========
print("Adding channel dummies...")
channel_dummies = pd.get_dummies(df_txn["channel_type"], prefix="channel")
df_txn = pd.concat([df_txn, channel_dummies], axis=1)
acct_channel = df_txn.groupby("from_acct")[channel_dummies.columns].mean()
acct_features = acct_features.merge(acct_channel, on="from_acct", how="left")

# ========== 時間特徵 ==========
print("Adding time features...")
df_txn["txn_time"] = pd.to_numeric(df_txn["txn_time"], errors="coerce")
df_txn["night"] = df_txn["txn_time"].apply(
    lambda t: (t >= 2200) or (t < 600) if pd.notna(t) else False
)

acct_features["active_days"] = df_txn.groupby("from_acct")["txn_date"].nunique()
acct_features["night_ratio"] = df_txn.groupby("from_acct")["night"].mean()

# 熵特徵
print("Adding entropy...")
counts = df_txn.groupby(["from_acct", "to_acct"]).size().reset_index(name="cnt")
total = counts.groupby("from_acct")["cnt"].transform("sum")
counts["p"] = counts["cnt"] / total
entropy_df = counts.groupby("from_acct").apply(
    lambda g: -(g["p"] * np.log2(g["p"])).sum()
)
acct_features["to_acct_entropy"] = entropy_df

In [75]:
# ========== Label ==========
print("Merging labels...")
df_alert = df_alert.rename(columns={"acct": "from_acct"})
train_data = acct_features.merge(df_alert, on="from_acct", how="left")
train_data["label"] = train_data["event_date"].notna().astype(int)
train_data = train_data.fillna(0)

print(train_data[["label"]].value_counts())

In [76]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

# ========= 切分資料 =========
X = train_data.drop(columns=["from_acct", "event_date", "label"], errors="ignore")
y = train_data["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ========= DMatrix =========
dtrain = xgb.DMatrix(X_train.astype("float32"), label=y_train)
dval = xgb.DMatrix(X_val.astype("float32"), label=y_val)

In [77]:
from xgboost.callback import TrainingCallback


class TQDMCallback(TrainingCallback):
    def __init__(self, total):
        self.pbar = tqdm(total=total, desc="Training XGBoost")

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False

    def after_training(self, model):
        self.pbar.close()
        return model


params = {
    "objective": "binary:logistic",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "aucpr",
    "seed": 42,
}

num_round = 300
evals = [(dtrain, "train"), (dval, "val")]

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    evals=evals,
    verbose_eval=False,
    callbacks=[TQDMCallback(num_round)],
)

In [78]:
import numpy as np
from sklearn.metrics import f1_score, classification_report, roc_auc_score

# 驗證集預測
y_proba = bst.predict(dval)

# 掃一遍 threshold
best_f1, best_th = 0, 0
for th in np.linspace(0.01, 0.99, 99):
    y_pred = (y_proba > th).astype(int)
    f1 = f1_score(y_val, y_pred)
    if f1 > best_f1:
        best_f1, best_th = f1, th

print(f"最佳 threshold: {best_th:.2f}, F1: {best_f1:.4f}")
print("ROC-AUC:", roc_auc_score(y_val, y_proba))

# 最佳 threshold 下的分類報告
y_pred = (y_proba > best_th).astype(int)
print(classification_report(y_val, y_pred))

In [79]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

precision, recall, thresholds = precision_recall_curve(y_val, y_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Validation)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import lightgbm as lgb
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc,
)
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import early_stopping, log_evaluation

# ========= 建立 LightGBM Dataset =========
dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

params = {
    "objective": "logitraw",
    "metric": ["auc", "average_precision"],  # AUC & PR-AUC
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "scale_pos_weight": scale_pos_weight,  # 跟 XGB 一樣處理 imbalance
    "seed": 42,
    "verbosity": -1,
}

num_round = 500
bst = lgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=50),
    ],
)

# ========= 驗證 =========
y_proba = bst.predict(X_val, num_iteration=bst.best_iteration)
y_pred = (y_proba > 0.5).astype(int)

print("分類報告 (threshold=0.5)：")
print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_proba))

# ========= 找最佳 threshold =========
prec, rec, thresholds = precision_recall_curve(y_val, y_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1_scores)
print("最佳 threshold:", thresholds[best_idx], "F1:", f1_scores[best_idx])

# ========= 畫 PR Curve =========
plt.figure()
plt.plot(rec, prec, label=f"PR AUC = {auc(rec, prec):.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Validation)")
plt.legend()
plt.show()

# ========= 特徵重要性 =========
lgb.plot_importance(bst, max_num_features=20, importance_type="gain")
plt.show()

In [82]:
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc,
    f1_score,
    precision_score,
    recall_score,
)
import numpy as np

# ========= LightGBM Dataset =========
dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

params = {
    "objective": "binary",
    "metric": ["auc", "average_precision"],  # AUC & PR-AUC
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "scale_pos_weight": scale_pos_weight,
    "seed": 42,
    "verbosity": -1,
}

num_round = 500
bst = lgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=50),
    ],
)

# ========= 驗證 =========
y_proba = bst.predict(X_val, num_iteration=bst.best_iteration)

# ROC / PR AUC
roc = roc_auc_score(y_val, y_proba)
prec, rec, thresholds = precision_recall_curve(y_val, y_proba)
pr_auc = auc(rec, prec)
print(f"ROC-AUC: {roc:.4f}, PR-AUC: {pr_auc:.4f}")

# ========= 找最佳 threshold =========
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1_scores)
best_th = thresholds[best_idx]
print(f"最佳 threshold: {best_th:.2f}, F1: {f1_scores[best_idx]:.4f}")

y_pred = (y_proba > best_th).astype(int)
print("\n分類報告 (最佳 threshold)：")
print(classification_report(y_val, y_pred))


# ========= Top-K 評估 =========
def evaluate_topk(y_true, y_proba, k_list=[100, 500, 1000, 2000, 5000]):
    results = {}
    sorted_idx = np.argsort(-y_proba)  # 機率由高到低排序
    for k in k_list:
        topk_idx = sorted_idx[:k]
        y_pred = np.zeros_like(y_true)
        y_pred[topk_idx] = 1
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        results[k] = {"precision": precision, "recall": recall, "f1": f1}
    return results


topk_results = evaluate_topk(y_val.values, y_proba)
print("\nTop-K 評估結果：")
for k, metrics in topk_results.items():
    print(f"Top-{k}: {metrics}")

In [83]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score


def evaluate_topk(y_true, y_proba, ks=[100, 500, 1000, 2000, 5000]):
    results = {}
    # 排序 (由高到低)
    idx = np.argsort(-y_proba)
    for k in ks:
        topk_idx = idx[:k]
        y_pred_topk = np.zeros_like(y_true)
        y_pred_topk[topk_idx] = 1
        prec = precision_score(y_true, y_pred_topk, zero_division=0)
        rec = recall_score(y_true, y_pred_topk, zero_division=0)
        f1 = 2 * prec * rec / (prec + rec + 1e-8)
        results[k] = {"precision": prec, "recall": rec, "f1": f1}
    return results


# === 計算 Top-K metrics ===
topk_results = evaluate_topk(
    y_val.values, y_proba, ks=[100, 500, 1000, 2000, 5000, 10000]
)

# === 繪圖 ===
plt.figure(figsize=(8, 5))
plt.plot(
    list(topk_results.keys()),
    [v["precision"] for v in topk_results.values()],
    marker="o",
    label="Precision",
)
plt.plot(
    list(topk_results.keys()),
    [v["recall"] for v in topk_results.values()],
    marker="o",
    label="Recall",
)
plt.plot(
    list(topk_results.keys()),
    [v["f1"] for v in topk_results.values()],
    marker="o",
    label="F1",
)
plt.xlabel("Top-K")
plt.ylabel("Score")
plt.title("Top-K Precision/Recall/F1")
plt.legend()
plt.grid(True)
plt.show()

# 印出數值
for k, v in topk_results.items():
    print(f"Top-{k}: {v}")

In [2]:
!uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

/bin/sh: uv: command not found


In [18]:
# ===== PyTorch Baseline: MLP + Focal Loss + Early Stopping (PR-AUC) =====
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
)
from dataclasses import dataclass
import math
import random
import time

# --------- 可調參數 ---------
SEED = 42
BATCH_SIZE = 2048
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 10  # Early stopping patience on PR-AUC
ALPHA = 0.25  # Focal loss alpha
GAMMA = 2.0  # Focal loss gamma
NUM_WORKERS = 0  # 設 0 以避免某些環境 DataLoader 卡住
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# --------- 固定隨機種子 ---------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


set_seed(SEED)

# --------- 檢查資料型態 ---------
assert isinstance(X_train, (pd.DataFrame, np.ndarray)) and isinstance(
    X_val, (pd.DataFrame, np.ndarray)
)
assert isinstance(y_train, (pd.Series, np.ndarray)) and isinstance(
    y_val, (pd.Series, np.ndarray)
)

# --------- 標準化（僅用 train fit，再 transform val）---------
scaler = StandardScaler()
X_train_np = scaler.fit_transform(np.asarray(X_train, dtype=np.float32))
X_val_np = scaler.transform(np.asarray(X_val, dtype=np.float32))
y_train_np = np.asarray(y_train, dtype=np.float32).reshape(-1, 1)
y_val_np = np.asarray(y_val, dtype=np.float32).reshape(-1, 1)


# --------- Dataset / DataLoader ---------
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = ArrayDataset(X_train_np, y_train_np)
val_ds = ArrayDataset(X_val_np, y_val_np)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# --------- 模型定義（簡潔 MLP）---------
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), p_drop=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(p_drop)]
            prev = h
        layers += [nn.Linear(prev, 1)]  # 單一輸出 logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)  # logits shape: (B,)


# --------- Focal Loss（對 logits 計算）---------
class FocalLoss(nn.Module):
    def __init__(self, alpha=ALPHA, gamma=GAMMA, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B,), targets: (B,) in {0,1}
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(
            -bce
        )  # pt = sigmoid(logits) 對於 y=1；1-sigmoid(logits) 對於 y=0
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        else:
            return loss


# --------- 訓練/驗證函數 ---------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).view(-1)
        logits = model(xb)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.detach().cpu().numpy())
        all_targets.append(yb.detach().cpu().numpy())
    y_prob = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_targets, axis=0)

    # 指標
    roc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan")
    pr = average_precision_score(y_true, y_prob)  # PR-AUC
    return y_prob, y_true, roc, pr


def train_loop(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    wd=WEIGHT_DECAY,
    patience=PATIENCE,
):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs, 10)
    )
    criterion = FocalLoss(alpha=ALPHA, gamma=GAMMA)

    best_pr = -1.0
    best_state = None
    best_epoch = -1
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n_sample = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True).view(-1)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            bs = xb.size(0)
            total_loss += loss.item() * bs
            n_sample += bs

        scheduler.step()
        train_loss = total_loss / max(n_sample, 1)

        # 驗證
        y_prob, y_true, roc, pr = evaluate(model, val_loader)

        print(
            f"[Epoch {epoch:03d}] train_loss={train_loss:.6f} | val_roc={roc:.6f} | val_pr={pr:.6f}"
        )

        # 早停（以 PR-AUC 為主）
        if pr > best_pr + 1e-8:
            best_pr = pr
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(
                    f"Early stopping at epoch {epoch}. Best PR-AUC={best_pr:.6f} (epoch {best_epoch})."
                )
                break

    # 載入最佳權重
    if best_state is not None:
        model.load_state_dict(best_state)
    # 最終在驗證集再評估一次
    y_prob, y_true, roc, pr = evaluate(model, val_loader)
    return model, y_prob, y_true, roc, pr, best_epoch, best_pr


# --------- 建模並訓練 ---------
input_dim = X_train_np.shape[1]
model = MLP(input_dim, hidden_dims=(256, 128), p_drop=0.3)
model, y_prob, y_true, roc, pr, best_epoch, best_pr = train_loop(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    wd=WEIGHT_DECAY,
    patience=PATIENCE,
)

print(
    f"\nFinal Validation: ROC-AUC={roc:.6f}, PR-AUC={pr:.6f} (best PR-AUC={best_pr:.6f} at epoch {best_epoch})"
)

# --------- 後處理：最佳閾值 F1、Top-K 指標 ---------
from sklearn.metrics import f1_score

prec, rec, th = precision_recall_curve(y_true, y_prob)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1s)
best_th = th[best_idx] if best_idx < len(th) else 0.5
best_f1 = f1s[best_idx]
print(f"Best threshold by F1 = {best_th:.6f}, F1 = {best_f1:.6f}")

# 固定 threshold=0.5 也列一次報告（方便對照）
y_pred_05 = (y_prob > 0.5).astype(int)
print("\nClassification report (threshold=0.5):")
print(classification_report(y_true, y_pred_05, digits=4, zero_division=0))

# 最佳 threshold 報告
y_pred_best = (y_prob > best_th).astype(int)
print("\nClassification report (best threshold):")
print(classification_report(y_true, y_pred_best, digits=4, zero_division=0))


# Top-K 評估
def eval_topk(y_true, y_score, ks=(100, 500, 1000, 2000, 5000)):
    idx = np.argsort(-y_score)
    res = {}
    P = y_true.sum()
    for k in ks:
        topk = idx[:k]
        tp = y_true[topk].sum()
        precision = tp / k
        recall = tp / P if P > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        res[f"Top-{k}"] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
        }
    return res


print("\nTop-K metrics:")
print(eval_topk(y_true.astype(int), y_prob))

[Epoch 001] train_loss=0.001424 | val_roc=0.784314 | val_pr=0.006387
[Epoch 002] train_loss=0.000621 | val_roc=0.790515 | val_pr=0.006872
[Epoch 003] train_loss=0.000583 | val_roc=0.810770 | val_pr=0.007346
[Epoch 004] train_loss=0.000581 | val_roc=0.800270 | val_pr=0.009568
[Epoch 005] train_loss=0.000565 | val_roc=0.812617 | val_pr=0.010407
[Epoch 006] train_loss=0.000561 | val_roc=0.850107 | val_pr=0.010685
[Epoch 007] train_loss=0.000567 | val_roc=0.855727 | val_pr=0.011589
[Epoch 008] train_loss=0.000560 | val_roc=0.848686 | val_pr=0.012221
[Epoch 009] train_loss=0.000559 | val_roc=0.856213 | val_pr=0.010528
[Epoch 010] train_loss=0.000549 | val_roc=0.856296 | val_pr=0.011328
[Epoch 011] train_loss=0.000549 | val_roc=0.859715 | val_pr=0.009215
[Epoch 012] train_loss=0.000549 | val_roc=0.851766 | val_pr=0.018364
[Epoch 013] train_loss=0.000548 | val_roc=0.850704 | val_pr=0.013765
[Epoch 014] train_loss=0.000551 | val_roc=0.863035 | val_pr=0.012305
[Epoch 015] train_loss=0.000551 | 